# Intro: Deterministic LLM Judge Patterns

Real teams often use judge-style rubrics. This notebook keeps the mechanics local by loading a rubric and fixture-backed judge outputs. Notice the `reasoning` field comes before the numeric score when we review each case.

In [ ]:
from pathlib import Path
import json
from pprint import pprint

from eval_harness.dataset import load_jsonl_cases
from eval_harness.judges import MockJudge, load_rubric
from eval_harness.metrics import is_valid_json

root = Path.cwd()
cases = load_jsonl_cases(root / "tests" / "evals" / "datasets" / "generation_cases.jsonl")
outputs = json.loads((root / "tests" / "evals" / "mocks" / "mock_model_outputs.json").read_text())
fixture_outputs = json.loads((root / "tests" / "evals" / "mocks" / "mock_judge_outputs.json").read_text())
rubric = load_rubric(root / "tests" / "evals" / "rubrics" / "answer_quality.yaml")
judge = MockJudge(fixture_outputs)

rows = []
for case in cases:
    actual = outputs[case.case_id]
    if case.metadata.get("requires_json"):
        rows.append(
            {
                "case_id": case.case_id,
                "reasoning": "Structured validation runs before rubric scoring.",
                "json_valid": is_valid_json(actual),
                "passed": False,
            }
        )
        continue
    judged = judge.judge(case, actual, rubric)
    rows.append(
        {
            "case_id": case.case_id,
            "reasoning": judged.reasoning,
            "scores": judged.scores,
            "passed": judged.passed,
        }
    )

pprint(rows)
